In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import torch
import torch.nn as nn
from torch.optim import SGD, Adam
from torch.nn import MSELoss, BCEWithLogitsLoss
from source.normal.data import trainLoader
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import ConcatDataset, DataLoader
from source.normal.model import EfficientModel
from source.normal.train import trainModel
from source.normal.loss import CustomLoss
from source.normal.optim import RAdam
from apex import amp

In [3]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/pretrain/train/'
    loader['label_path'] = '../../data/pretrain/train_folds.csv'
    loader['size'] = 256
    loader['fold_idx'] = fold
    train_1, valid_1 = trainLoader(**loader)
    loader = {}
    loader['image_path'] = '../../data/pretrain/test/'
    loader['label_path'] = '../../data/pretrain/test_folds.csv'
    loader['size'] = 256
    loader['fold_idx'] = fold
    train_2, valid_2 = trainLoader(**loader)
    train = ConcatDataset([train_1, train_2])
    valid = ConcatDataset([valid_1, valid_2])
    train = DataLoader(train, batch_size=20, shuffle=True, num_workers=6, drop_last=True)
    valid = DataLoader(valid, batch_size=6, shuffle=True, num_workers=6, drop_last=True)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = RAdam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    schedular = StepLR(optimizer, step_size=2, gamma=0.5)
    model, optimizer = amp.initialize(model, optimizer, opt_level="O2",keep_batchnorm_fp32=True, verbosity=0)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = CustomLoss(weight=0.75, variance=0.) 
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/pretrain/model_{}.pt'.format(fold)
    trainer['epochs'] = 10
    trainer['batch'] = 20
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(1)

Train Images: 31611 Valid Images: 3515
Train Images: 48215 Valid Images: 5361
Loaded pretrained weights for efficientnet-b3


100% 79820/79820 [14:08<00:00, 94.11it/s, trn_ls=0.81639, trn_mt=0.56276, val_ls=0.67680, val_mt=0.70332] 
/opt/conda/lib/python3.7/site-packages/torch/optim/lr_scheduler.py:73: UserWarning: Seems like `optimizer.step()` has been overridden after learning rate scheduler initialization. Please, make sure to call `optimizer.step()` before `lr_scheduler.step()`. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  "https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate", UserWarning)
100% 79820/79820 [14:10<00:00, 93.87it/s, trn_ls=0.62518, trn_mt=0.71690, val_ls=0.63402, val_mt=0.73214] 
100% 79820/79820 [14:04<00:00, 94.48it/s, trn_ls=0.56835, trn_mt=0.76860, val_ls=0.59828, val_mt=0.77678] 
100% 79820/79820 [14:09<00:00, 93.99it/s, trn_ls=0.54757, trn_mt=0.78553, val_ls=0.59267, val_mt=0.77519] 
100% 79820/79820 [14:03<00:00, 94.58it/s, trn_ls=0.52409, trn_mt=0.80366, val_ls=0.58752, val_mt=0.78535] 
100% 79820/79820 [14:14<00:00,